In [3]:
import pandas as pd
import numpy as np
import importlib
from decimal import *
import sys
import os
hostname = os.uname().nodename
if hostname == 'BlackBeast':
    path = '/home/hedvigs/snake_book/econ'
    site = 'home'
elif hostname == 'hedvig-hp-elitedesk-800-g5-twr':
    path = '/home/hedvigs/PycharmProjects/homewrs/plab_workflow'
    site = 'work'
elif hostname == 'work-computer':
    path = '/mnt/work/workbench/hedvigs/snake_book/econ'
    site = 'server'
elif hostname == 'SilverFlex':
    path = '/home/hedvigs/gitrepos/plab_workflow'
    site = 'silverFlex'

sys.path.append(path)
print(path)

from workflow.scripts.data_management import setup_data as gt
from workflow.scripts.tables.make_tex_table import aggregate_df
from workflow.scripts.tables import table_functions as tf
#from workflow.scripts.data_management import subsets


/home/hedvigs/gitrepos/plab_workflow


In [5]:
# read in selected SNPs
snps_m = pd.read_csv(path + '/results/report/selected_snps_m.csv', sep='\t')
snps_f = pd.read_csv(path + '/results/report/selected_snps_f.csv', sep='\t')
#snps_c = pd.read_csv(path + '/results/report/selected_snps_combine.csv', sep='\t')

In [6]:
def rename_snps(snp_list_c):
    suffixes = ["_f", "_m", "_2"]
    snp_list = pd.DataFrame(index=snp_list_c.index)
    for fold in snp_list_c.columns:
        snp_list[fold] = snp_list_c[fold].apply(
            lambda snp: 
                snp.removesuffix(suffixes[0])
                if 
                snp.endswith(suffixes[0])
                else 
                snp.removesuffix(suffixes[1])
                if 
                snp.endswith(suffixes[1]) 
                else 
                snp.removesuffix(suffixes[2])
                if 
                snp.endswith(suffixes[2]) 
                else
                'rs113698135' 
                if
                snp == 'chr17:13771093_C/A'
                else
                snp
                )
    return snp_list

In [7]:
# rename snps with suffixes or names that dont correspond to the indep.clumped list, positions are equal
#snps_c =rename_snps(snps_c)
snps_f = rename_snps(snps_f)
snps_m = rename_snps(snps_m)

In [8]:
# get chr and p-value from indep.clumped list
snp_list_pval = pd.read_csv(path + '/config/snp_list_pval.csv', sep='\t', index_col='SNP')
snp_list_pval.drop(columns=['Unnamed: 0'], inplace=True)
snp_list_pval.CHR = snp_list_pval.CHR.astype(str)
# change to exp format
pd.set_option('display.float_format', '{:.2e}'.format)
snp_list_pval

,CHR,P
SNP,,
rs2963463,5,2.44e-47
rs12037376,1,2.02e-24
rs5991030,23,2.81e-24
rs4359773,3,2.53e-22
rs34555419,2,3.41e-17
...,...,...
rs141382777,16,9.98e-01
rs72775770,16,9.98e-01
rs6472755,8,9.99e-01


In [12]:
# maternal snps
snps_0m = snps_m['0'].to_list()
snps_1m = snps_m['1'].to_list()
snps_2m = snps_m['2'].to_list()
snps_3m = snps_m['3'].to_list()
snps_4m = snps_m['4'].to_list()
# fetal snps
snps_0f = snps_f['0'].to_list()
snps_1f = snps_f['1'].to_list()
snps_2f = snps_f['2'].to_list()
snps_3f = snps_f['3'].to_list()
snps_4f = snps_f['4'].to_list()
# combined snps
# snps_0c = snps_c['0'].to_list()
# snps_1c = snps_c['1'].to_list()
# snps_2c = snps_c['2'].to_list()
# snps_3c = snps_c['3'].to_list()
# snps_4c = snps_c['4'].to_list()

In [9]:
snps_m

,0,1,2,3,4
0,rs35836191,rs72659201,rs7538997,rs76266717,rs4655235
1,rs116825850,rs150794748,rs116825850,rs35836191,rs35836191
2,rs150794748,rs76677301,rs76677301,rs72659201,rs880466
3,rs664777,rs664777,rs76035212,rs187509586,rs61769198
4,rs2075995,rs11589432,rs71638874,rs10917196,rs139831675
...,...,...,...,...,...
95,rs2001124,rs75379427,rs78336371,rs77764993,rs75379427
96,rs149928287,rs138458916,rs117114983,rs56942423,rs76604150
97,rs144545499,rs186125715,rs141594674,rs75379427,rs80019664
98,rs140384340,rs145863381,rs139426544,rs75451809,rs140384340


## Maternal genome

In [18]:
# create a score based on rankings for different folds
# maternal genome
snps_m.index.name="Rank"
snps_m.reset_index(inplace=True)

# reverse the rank to create the score
snps_m['Score'] = 100-snps_m["Rank"]
# set SNPs as index to find
snpr_0 = snps_m.set_index("0")
snpr_1 = snps_m.set_index("1")
snpr_2 = snps_m.set_index("2")
snpr_3 = snps_m.set_index("3")
snpr_4 = snps_m.set_index("4")

In [19]:
# calculate the sum of score for each snp
snp_m = pd.DataFrame()
all_snps = set(snps_0m + snps_1m + snps_2m + snps_3m + snps_4m )
snp_m['SNP'] = list(all_snps)
snp_m['Chromosome'] = snp_m['SNP'].apply(lambda x: snp_list_pval.loc[x, 'CHR'] if x in snp_list_pval.index else np.nan)
snp_m['p-value'] = snp_m['SNP'].apply(lambda x: snp_list_pval.loc[x, 'P'] if x in snp_list_pval.index else np.nan)

snp_m['score0'] = snp_m['SNP'].apply(lambda x: snpr_0.loc[x, "Score"] if x in snpr_0.index else 0)
snp_m['score1'] = snp_m['SNP'].apply(lambda x: snpr_1.loc[x, "Score"] if x in snpr_1.index else 0)
snp_m['score2'] = snp_m['SNP'].apply(lambda x: snpr_2.loc[x, "Score"] if x in snpr_2.index else 0)
snp_m['score3'] = snp_m['SNP'].apply(lambda x: snpr_3.loc[x, "Score"] if x in snpr_3.index else 0)
snp_m['score4'] = snp_m['SNP'].apply(lambda x: snpr_4.loc[x, "Score"] if x in snpr_4.index else 0)
score_cols = ['score0', 'score1', 'score2', 'score3', 'score4']
snp_m['score_sum'] = snp_m[score_cols].apply(lambda row: np.sum(row), axis=1)
snp_m.sort_values('score_sum', ascending=False, inplace=True)
snp_m.reset_index(inplace=True, drop=True) 
#snp_m.set_index('score_sum', inplace=True)
snp_m.index.name="Rank"
snp_m.reset_index(inplace=True)
snp_m.set_index('SNP', inplace=True)


In [ ]:
# Only maternal
snp_presence = pd.DataFrame()
all_snps = set(snps_0m + snps_1m + snps_2m + snps_3m + snps_4m )
snp_presence['SNP'] = list(all_snps)
snp_presence['Chromosome'] = snp_presence['SNP'].apply(lambda x: snp_list_pval.loc[x, 'CHR'] if x in snp_list_pval.index else np.nan)
snp_presence['p-value'] = snp_presence['SNP'].apply(lambda x: snp_list_pval.loc[x, 'P'] if x in snp_list_pval.index else np.nan)
snp_presence['Rank'] = snp_presence['SNP'].apply(lambda x: snp_m.loc[x, "Rank"] if x in snp_m.index else np.nan)
# Determine presence in each group and assign values
snp_presence['0'] = snp_presence['SNP'].apply(lambda x: 1 if x in snps_0m else 0)
snp_presence['1'] = snp_presence['SNP'].apply(lambda x: 1 if x in snps_1m else 0)
snp_presence['2'] = snp_presence['SNP'].apply(lambda x: 1 if x in snps_2m else 0)
snp_presence['3'] = snp_presence['SNP'].apply(lambda x: 1 if x in snps_3m else 0)
snp_presence['4'] = snp_presence['SNP'].apply(lambda x: 1 if x in snps_4m else 0)

fold_cols = ['0', '1', '2', '3', '4']
snp_presence['Total'] = snp_presence[fold_cols].apply(lambda row: sum(row), axis=1)
snp_presence.sort_values('p-value', inplace=True)
snp_presence.set_index("Rank", inplace=True)
snp_presence_m = snp_presence
snp_presence_m[snp_presence_m.Total==5]

,SNP,Chromosome,p-value,0,1,2,3,4,Total
Rank,,,,,,,,,
1,rs34555419,2,3.41e-17,1,1,1,1,1,5
64,rs7023208,9,2.60e-12,1,1,1,1,1,5
10,rs79220368,4,6.54e-03,1,1,1,1,1,5
16,rs2844580,6,1.61e-02,1,1,1,1,1,5
46,rs72716936,8,7.90e-02,1,1,1,1,1,5
2,rs9854485,3,8.77e-02,1,1,1,1,1,5
53,rs72718778,8,1.60e-01,1,1,1,1,1,5
223,rs56942423,17,2.04e-01,1,1,1,1,1,5
6,rs149527606,4,2.24e-01,1,1,1,1,1,5


In [ ]:
importlib.reload(tf)
filename_m = "selectedSNPsMaternal"
tf.save_df_to_tex(snp_presence_m, filename=filename_m, site=site )

## Fetal genome

In [21]:
# create a score based on rankings for different folds
# fetal genome
snps_f.index.name="Rank"
snps_f.reset_index(inplace=True)

# reverse the rank to create the score
snps_f['Score'] = 100-snps_f["Rank"]
# set SNPs as index to find
snpr_0 = snps_f.set_index("0")
snpr_1 = snps_f.set_index("1")
snpr_2 = snps_f.set_index("2")
snpr_3 = snps_f.set_index("3")
snpr_4 = snps_f.set_index("4")

In [22]:
# calculate the sum of score for each snp
# fetal
snp_f = pd.DataFrame()
all_snps = set(snps_0f + snps_1f + snps_2f + snps_3f + snps_4f)
snp_f['SNP'] = list(all_snps)

snp_f['score0'] = snp_f['SNP'].apply(lambda x: snpr_0.loc[x, "Score"] if x in snpr_0.index else 0)
snp_f['score1'] = snp_f['SNP'].apply(lambda x: snpr_1.loc[x, "Score"] if x in snpr_1.index else 0)
snp_f['score2'] = snp_f['SNP'].apply(lambda x: snpr_2.loc[x, "Score"] if x in snpr_2.index else 0)
snp_f['score3'] = snp_f['SNP'].apply(lambda x: snpr_3.loc[x, "Score"] if x in snpr_3.index else 0)
snp_f['score4'] = snp_f['SNP'].apply(lambda x: snpr_4.loc[x, "Score"] if x in snpr_4.index else 0)
score_cols = ['score0', 'score1', 'score2', 'score3', 'score4']
snp_f['score_sum'] = snp_f[score_cols].apply(lambda row: np.sum(row), axis=1)
snp_f.sort_values('score_sum', ascending=False, inplace=True)
snp_f.reset_index(inplace=True, drop=True) 

snp_f.index.name="Rank"
snp_f.reset_index(inplace=True)
snp_f.set_index('SNP', inplace=True)


In [23]:
# Only fetal
snp_presence = pd.DataFrame()
all_snps = set(snps_0f + snps_1f + snps_2f + snps_3f + snps_4f)
snp_presence['SNP'] = list(all_snps)
snp_presence['Chromosome'] = snp_presence['SNP'].apply(lambda x: snp_list_pval.loc[x, 'CHR'] if x in snp_list_pval.index else np.nan)
snp_presence['p-value'] = snp_presence['SNP'].apply(lambda x: snp_list_pval.loc[x, 'P'] if x in snp_list_pval.index else np.nan)
snp_presence['Rank'] = snp_presence['SNP'].apply(lambda x: snp_f.loc[x, "Rank"] if x in snp_f.index else np.nan)
# Determine presence in each group and assign values
snp_presence['0'] = snp_presence['SNP'].apply(lambda x: 1 if x in snps_0f else 0)
snp_presence['1'] = snp_presence['SNP'].apply(lambda x: 1 if x in snps_1f else 0)
snp_presence['2'] = snp_presence['SNP'].apply(lambda x: 1 if x in snps_2f else 0)
snp_presence['3'] = snp_presence['SNP'].apply(lambda x: 1 if x in snps_3f else 0)
snp_presence['4'] = snp_presence['SNP'].apply(lambda x: 1 if x in snps_4f else 0)

fold_cols = ['0', '1', '2', '3', '4']
snp_presence['Total'] = snp_presence[fold_cols].apply(lambda row: sum(row), axis=1)
snp_presence.sort_values('p-value', inplace=True)
snp_presence.set_index("Rank", inplace=True)
snp_presence_f = snp_presence
snp_presence_f[snp_presence_f.Total==5]


,SNP,Chromosome,p-value,0,1,2,3,4,Total
Rank,,,,,,,,,
24,rs118027082,8,1.81e-02,1,1,1,1,1,5
3,rs62233032,3,4.11e-02,1,1,1,1,1,5
17,rs2495991,6,4.89e-02,1,1,1,1,1,5
5,rs145590853,3,7.76e-02,1,1,1,1,1,5
0,rs71638874,1,1.36e-01,1,1,1,1,1,5
8,rs77217171,4,2.40e-01,1,1,1,1,1,5
102,rs146923460,17,2.71e-01,1,1,1,1,1,5
63,rs116875306,16,3.00e-01,1,1,1,1,1,5
15,rs438475,6,3.69e-01,1,1,1,1,1,5


In [18]:
importlib.reload(tf)
filename_f = "selectedSNPsFetal"
tf.save_df_to_tex(snp_presence_f, filename=filename_f, site=site )

286
9
0
Table saved
header 
[]

 Rank & 
['\n', 'midrule\n', 'endfirsthead\n', 'caption[]{selectedSNPsFetal} ']
['endhead\n', 'midrule\n', 'multicolumn{10}{r}{Continued on next page} ', '']
Table saved and customized.


## Combined if needed 

In [ ]:
## comb snps
# snps in both maternal and fetal
snps_0mf = [snp for snp in snps_0m if snp in snps_0f]
snps_1mf = [snp for snp in snps_1m if snp in snps_1f]
snps_2mf = [snp for snp in snps_2m if snp in snps_2f]
snps_3mf = [snp for snp in snps_3m if snp in snps_3f]
snps_4mf = [snp for snp in snps_4m if snp in snps_4f]
# snps in both maternal and combined
snps_0mc = [snp for snp in snps_0m if snp in snps_0c]
snps_1mc = [snp for snp in snps_1m if snp in snps_1c]
snps_2mc = [snp for snp in snps_2m if snp in snps_2c]
snps_3mc = [snp for snp in snps_3m if snp in snps_3c]
snps_4mc = [snp for snp in snps_4m if snp in snps_4c]
# snps in both fetal and combined
snps_0fc = [snp for snp in snps_0f if snp in snps_0c]
snps_1fc = [snp for snp in snps_1f if snp in snps_1c]
snps_2fc = [snp for snp in snps_2f if snp in snps_2c]
snps_3fc = [snp for snp in snps_3f if snp in snps_3c]
snps_4fc = [snp for snp in snps_4f if snp in snps_4c]
# snps in all three
snps_0mfc = [snp for snp in snps_0mf if snp in snps_0c]
snps_1mfc = [snp for snp in snps_1mf if snp in snps_1c]
snps_2mfc = [snp for snp in snps_2mf if snp in snps_2c]
snps_3mfc = [snp for snp in snps_3mf if snp in snps_3c]
snps_4mfc = [snp for snp in snps_4mf if snp in snps_4c]

In [1]:
#maternal fetal
snp_presence = pd.DataFrame()
all_snps = set(snps_0m + snps_1m + snps_2m + snps_3m + snps_4m )#+ snps_0f + snps_1f + snps_2f + snps_3f + snps_4f)# + snps_0c + snps_1c + snps_2c + snps_3c + snps_4c)
snp_presence['SNP'] = list(all_snps)
snp_presence['Chromosome'] = snp_presence['SNP'].apply(lambda x: snp_list_pval.loc[x, 'CHR'] if x in snp_list_pval.index else np.nan)
snp_presence['p-value'] = snp_presence['SNP'].apply(lambda x: snp_list_pval.loc[x, 'P'] if x in snp_list_pval.index else np.nan)

# Determine presence in each group and assign values
snp_presence['0'] = snp_presence['SNP'].apply(lambda x: 'mf' if x in snps_0mf else 'm' if x in snps_0m else 'f' if x in snps_0f else 0)
snp_presence['1'] = snp_presence['SNP'].apply(lambda x: 'mf' if x in snps_1mf else 'm' if x in snps_1m else 'f' if x in snps_1f else 0)
snp_presence['2'] = snp_presence['SNP'].apply(lambda x: 'mf' if x in snps_2mf else 'm' if x in snps_2m else 'f' if x in snps_2f else 0)
snp_presence['3'] = snp_presence['SNP'].apply(lambda x: 'mf' if x in snps_3mf else 'm' if x in snps_3m else 'f' if x in snps_3f else 0)
snp_presence['4'] = snp_presence['SNP'].apply(lambda x: 'mf' if x in snps_4mf else 'm' if x in snps_4m else 'f' if x in snps_4f else 0)

snp_presence['Total'] = snp_presence[['0', '1', '2', '3', '4']].apply(lambda row: sum(3 for val in row if val == 'mfc') + sum(2 for val in row if val in ['mf', 'mc', 'fc']) + sum(1 for val in row if val in ['m', 'f', 'c']), axis=1)
#snp_presence.drop(columns=['Unnamed: 0'], inplace=True)
snp_presence.head()
snp_presence.sort_values('p-value', inplace=True)
#snp_presence.reset_index(inplace=True, drop=True)
snp_presence.index.name="Rank"
snp_presence[snp_presence.Total==5]

NameError: name 'pd' is not defined

In [ ]:
# maternal fetal combined
snp_presence = pd.DataFrame()
all_snps = set(snps_0m + snps_1m + snps_2m + snps_3m + snps_4m + snps_0f + snps_1f + snps_2f + snps_3f + snps_4f)# + snps_0c + snps_1c + snps_2c + snps_3c + snps_4c)
snp_presence['SNP'] = list(all_snps)
snp_presence['Chromosome'] = snp_presence['SNP'].apply(lambda x: snp_list_pval.loc[x, 'CHR'] if x in snp_list_pval.index else np.nan)
snp_presence['p-value'] = snp_presence['SNP'].apply(lambda x: snp_list_pval.loc[x, 'P'] if x in snp_list_pval.index else np.nan)

# Determine presence in each group and assign values
snp_presence['0'] = snp_presence['SNP'].apply(lambda x: 'mfc' if x in snps_0mfc else 'mf' if x in snps_0mf else 'mc' if x in snps_0mc else 'fc' if x in snps_0fc else 'm' if x in snps_0m else 'f' if x in snps_0f else 'c' if x in snps_0c else 0)
snp_presence['1'] = snp_presence['SNP'].apply(lambda x: 'mfc' if x in snps_1mfc else 'mf' if x in snps_1mf else 'mc' if x in snps_1mc else 'fc' if x in snps_1fc else 'm' if x in snps_1m else 'f' if x in snps_1f else 'c' if x in snps_1c else 0)
snp_presence['2'] = snp_presence['SNP'].apply(lambda x: 'mfc' if x in snps_2mfc else 'mf' if x in snps_2mf else 'mc' if x in snps_2mc else 'fc' if x in snps_2fc else 'm' if x in snps_2m else 'f' if x in snps_2f else 'c' if x in snps_2c else 0)
snp_presence['3'] = snp_presence['SNP'].apply(lambda x: 'mfc' if x in snps_3mfc else 'mf' if x in snps_3mf else 'mc' if x in snps_3mc else 'fc' if x in snps_3fc else 'm' if x in snps_3m else 'f' if x in snps_3f else 'c' if x in snps_3c else 0)
snp_presence['4'] = snp_presence['SNP'].apply(lambda x: 'mfc' if x in snps_4mfc else 'mf' if x in snps_4mf else 'mc' if x in snps_4mc else 'fc' if x in snps_4fc else 'm' if x in snps_4m else 'f' if x in snps_4f else 'c' if x in snps_4c else 0)

snp_presence['Total'] = snp_presence[['0', '1', '2', '3', '4']].apply(lambda row: sum(3 for val in row if val == 'mfc') + sum(2 for val in row if val in ['mf', 'mc', 'fc']) + sum(1 for val in row if val in ['m', 'f', 'c']), axis=1)
#snp_presence.drop(columns=['Unnamed: 0'], inplace=True)
snp_presence.head()
snp_presence.sort_values('p-value', inplace=True)
#snp_presence.reset_index(drop=True, inplace=True)
#snp_presence.set_index('chr', inplace=True)
#snp_presence.droplevel(0)
#snp_presence[snp_presence.chr.isna()]
snp_presence.reset_index(inplace=True, drop=True)
snp_presence.index.name="Rank"

In [ ]:
importlib.reload(tf)
filename_c = "selectedSNPsCombine"
tf.save_df_to_tex(snp_presence_c, filename=filename_c, site=site )

299
9
0
Table saved
header 
[]

 Rank & 
['\n', 'midrule\n', 'endfirsthead\n', 'caption[]{selectedSNPsMaternal} ']
['endhead\n', 'midrule\n', 'multicolumn{10}{r}{Continued on next page} ', '']
Table saved and customized.
